
# Silver Clientes

**Fuente:** `prod_dataco.brz_dataco.raw_dataco_supply_chain` (histórico completo acumulado)

**Destino:** Tabla de **Clientes**, en `prod_dataco.slv_dataco`

**Grano:** un registro por `customer_id`

## Qué hace

1. Lee todo el histórico de Bronze acumulado, no solo el último corte.
2. Deduplica por `customer_id` — se conserva una fila por cliente, manteniendo el mismo criterio del SP original (`ORDER BY (SELECT NULL)`).
3. Limpia y normaliza los datos:

   * Campos de texto: `trim` + mayúsculas.
   * Identificadores numéricos: `cast` a `int`.
4. Conserva las columnas de trazabilidad provenientes de Bronze.
5. Escribe la tabla Silver mediante `overwrite` completo, equivalente al `TRUNCATE` + `INSERT` del SP original.

## Trazabilidad conservada

* `fecha_inicio_corte`
* `fecha_fin_corte`
* `fecha_carga`
* `id_ejecucion`


## Pendiente / conocido

El criterio de deduplicación mantiene el comportamiento del SP original y no prioriza una versión específica del registro.

Si existen múltiples registros para un mismo `customer_id` con valores diferentes, la fila retenida puede variar entre ejecuciones.

In [0]:
# ============================================================
# Celda 1: Parámetros + lectura de Bronze
# ------------------------------------------------------------
# Define los widgets del notebook (catálogo, esquema y tabla
# Bronze/Silver) y lee TODO el histórico acumulado en Bronze
# (no solo el último corte), porque silver_clientes se
# reconstruye completo cada corrida (mismo criterio que el
# TRUNCATE+dedup del SP original).
# ============================================================

dbutils.widgets.text("bronze_catalog", "prod_dataco", "Catálogo Bronze")
dbutils.widgets.text("bronze_schema", "brz_dataco", "Esquema Bronze")
dbutils.widgets.text("bronze_table", "raw_dataco_supply_chain", "Tabla Bronze")

dbutils.widgets.text("silver_catalog", "prod_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "slv_dataco", "Esquema Silver")

bronze_catalog = dbutils.widgets.get("bronze_catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
bronze_table = dbutils.widgets.get("bronze_table")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")

BRONZE_TABLE_FQN = f"{bronze_catalog}.{bronze_schema}.{bronze_table}"
SILVER_TABLE_FQN = f"{silver_catalog}.{silver_schema}.md_clientes"

df_bronze = spark.table(BRONZE_TABLE_FQN)

print(f"Total filas en Bronze: {df_bronze.count()}")
print(f"Origen: {BRONZE_TABLE_FQN}")
print(f"Destino: {SILVER_TABLE_FQN}")

In [0]:
# ============================================================
# Celda 2: Helpers de limpieza + deduplicación
# ------------------------------------------------------------
# clean_str  -> equivalente a UPPER(TRIM(ISNULL(col, '')))
# clean_num  -> equivalente a CAST(ISNULL(NULLIF(TRIM(col),''),'0') AS tipo)
# dedup      -> equivalente a ROW_NUMBER() OVER (PARTITION BY ... ORDER BY
#               (SELECT NULL)) = 1 del SP original: se queda con UNA fila
#               arbitraria por Customer Id (no la más reciente, el mismo
#               criterio del SP)
# ============================================================

from pyspark.sql import functions as F, Window

def clean_str(col_name: str, default: str = "") -> F.Column:
    return F.upper(F.trim(F.coalesce(F.col(col_name).cast("string"), F.lit(default))))

def clean_num(col_name: str, cast_type, default: str = "0") -> F.Column:
    trimmed = F.trim(F.col(col_name).cast("string"))
    nulled = F.when((trimmed.isNull()) | (trimmed == ""), None).otherwise(trimmed)
    return F.coalesce(nulled, F.lit(default)).cast(cast_type)

def dedup(df, partition_cols: list):
    w = Window.partitionBy(*partition_cols).orderBy(F.monotonically_increasing_id())
    return (
        df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

print("✓ Helpers definidos: clean_str, clean_num, dedup")

In [0]:
[c for c in df_bronze.columns if "customer" in c.lower()]

In [0]:
# ============================================================
# Celda 3: Deduplicación + construcción de md_clientes
# ------------------------------------------------------------
# 1) dedup por customer_id -> una sola fila por cliente
#    (mismo criterio del SP: ORDER BY (SELECT NULL), arbitrario)
# 2) Aplica clean_num al id (INT) y clean_str al resto
# 3) Utiliza nomenclatura estándar snake_case para Silver
# 4) Conserva las columnas de trazabilidad provenientes de Bronze
# ============================================================

df_clientes_dedup = dedup(df_bronze, ["customer_id"])

df_silver_clientes = df_clientes_dedup.select(
    clean_num("customer_id", "int").alias("customer_id"),
    clean_str("customer_fname").alias("customer_fname"),
    clean_str("customer_lname").alias("customer_lname"),
    clean_str("customer_email").alias("customer_email"),
    clean_str("customer_password").alias("customer_password"),
    clean_str("customer_segment").alias("customer_segment"),
    clean_str("customer_city").alias("customer_city"),
    clean_str("customer_state").alias("customer_state"),
    clean_str("customer_street").alias("customer_street"),
    clean_str("customer_zipcode").alias("customer_zipcode"),
    clean_str("customer_country").alias("customer_country"),

    # --------------------------------------------------------
    # Trazabilidad
    # --------------------------------------------------------
    F.col("fecha_inicio_corte"),
    F.col("fecha_fin_corte"),
    F.col("fecha_carga"),
    F.col("id_ejecucion")
)

print(f"Filas antes del dedup : {df_bronze.count()}")
print(f"Filas después del dedup : {df_silver_clientes.count()}")

display(df_silver_clientes.limit(10))

In [0]:
# ============================================================
# Celda 3b: Crear esquema Silver (el catálogo prod_dataco ya existe)
# ============================================================

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{silver_catalog}.{silver_schema}"
)

print(f"✓ Esquema {silver_catalog}.{silver_schema} listo")

In [0]:
# ============================================================
# MERGE EN SILVER
# Replica el comportamiento de usp_tskInsDataMart_Clientes
# ============================================================

from delta.tables import DeltaTable

if not spark.catalog.tableExists(TABLA_SILVER):
    (
        df_clientes.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TABLA_SILVER)
    )

    print(f"✓ Tabla {TABLA_SILVER} creada inicialmente")

else:
    delta_target = DeltaTable.forName(spark, TABLA_SILVER)

    (
        delta_target.alias("t")
        .merge(
            df_clientes.alias("s"),
            "t.customer_id = s.customer_id"
        )
        .whenMatchedUpdate(set={
            "customer_fname": "s.customer_fname",
            "customer_lname": "s.customer_lname",
            "customer_email": "s.customer_email",
            "customer_segment": "s.customer_segment",
            "customer_city": "s.customer_city",
            "customer_state": "s.customer_state",
            "customer_street": "s.customer_street",
            "customer_zipcode": "s.customer_zipcode",
            "customer_country": "s.customer_country",

            "fecha_inicio_corte": "s.fecha_inicio_corte",
            "fecha_fin_corte": "s.fecha_fin_corte",
            "fecha_carga": "s.fecha_carga",
            "id_ejecucion": "s.id_ejecucion"
        })
        .whenNotMatchedInsert(values={
            "customer_id": "s.customer_id",
            "customer_fname": "s.customer_fname",
            "customer_lname": "s.customer_lname",
            "customer_email": "s.customer_email",
            "customer_password": "s.customer_password",
            "customer_segment": "s.customer_segment",
            "customer_city": "s.customer_city",
            "customer_state": "s.customer_state",
            "customer_street": "s.customer_street",
            "customer_zipcode": "s.customer_zipcode",
            "customer_country": "s.customer_country",

            "fecha_inicio_corte": "s.fecha_inicio_corte",
            "fecha_fin_corte": "s.fecha_fin_corte",
            "fecha_carga": "s.fecha_carga",
            "id_ejecucion": "s.id_ejecucion"
        })
        .execute()
    )

    print(f"✓ MERGE ejecutado sobre {TABLA_SILVER}")

In [0]:
%sql
SELECT *
FROM slv_dataco.operaciones.md_clientes
LIMIT 5;